In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve path for train.csv robustly in both root and notebooks execution contexts
root_dir = Path.cwd()
if root_dir.name == 'notebooks':
    root_dir = root_dir.parent

data_path = root_dir / 'data' / 'train.csv'
print('Using train data file:', data_path)
train_df = pd.read_csv(data_path)

# Handle missing data
numeric_cols = train_df.select_dtypes(include=[np.number]).columns
categorical_cols = ['gender', 'discharge_day_of_week', 'insurance_type']

print(f"Target distribution: \n{train_df['readmitted_30d'].value_counts(normalize=True)}")

Using train data file: /home/grow-lt-360/Downloads/DL_TEST/data/train.csv
Target distribution: 
readmitted_30d
0    0.91
1    0.09
Name: proportion, dtype: float64


In [7]:
import torch.nn as nn

class ReadmissionPredictor(nn.Module):
    def __init__(self, input_dim):
        super(ReadmissionPredictor, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(32, 1)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.drop2(x)
        x = self.fc3(x)
        return x

print("Model Arch defined.")

Model Arch defined.


In [13]:
import os
import joblib
import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Prepare directories
model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)

# Preprocessing
df_processed = train_df.copy()
y = df_processed['readmitted_30d'].values

# Fill missing values
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.drop(['readmitted_30d'])
df_processed[numeric_cols] = df_processed[numeric_cols].fillna(df_processed[numeric_cols].median())

# Handle categoricals
for col in categorical_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].fillna(df_processed[col].mode()[0])
        
df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)

# Drop identifier and date columns
cols_to_drop = ['patient_id', 'admission_date', 'readmitted_30d']
X = df_processed.drop(columns=[c for c in cols_to_drop if c in df_processed.columns])

# Store train columns for alignment during inference
train_columns = X.columns.tolist()
joblib.dump(train_columns, os.path.join(model_dir, 'train_columns.pkl'))

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
joblib.dump(scaler, os.path.join(model_dir, 'scaler.pkl'))

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

# Initialize model
input_dim = X_train_tensor.shape[1]
model = ReadmissionPredictor(input_dim)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 100
batch_size = 64
n_batches = len(X_train_tensor) // batch_size

best_val_auc = 0

for epoch in range(epochs):
    model.train()
    for i in range(n_batches):
        start_i = i * batch_size
        end_i = start_i + batch_size
        batch_X = X_train_tensor[start_i:end_i]
        batch_y = y_train_tensor[start_i:end_i]
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
    # Validation
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_preds = (torch.sigmoid(val_outputs) >= 0.5).float()
        val_loss = criterion(val_outputs, y_val_tensor)
        val_acc = accuracy_score(y_val, val_preds.numpy())
        val_precision = precision_score(y_val, val_preds.numpy())
        val_recall = recall_score(y_val, val_preds.numpy())
        val_f1 = f1_score(y_val, val_preds.numpy())
        val_auc = roc_auc_score(y_val, torch.sigmoid(val_outputs).numpy())
        
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), os.path.join(model_dir, 'best_model.pth'))
            
    if (epoch+1) % 10 == 0:
        print(f'''
Epoch {epoch+1}/{epochs}
Loss: {val_loss.item():.4f}
Accuracy: {val_acc:.4f}
Precision: {val_precision:.4f}
Recall: {val_recall:.4f}
F1 Score: {val_f1:.4f}
AUROC: {val_auc:.4f}
''')



/home/grow-lt-360/Downloads/DL_TEST/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/grow-lt-360/Downloads/DL_TEST/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Epoch 10/100
Loss: 0.1552
Accuracy: 0.9447
Precision: 0.7955
Recall: 0.5147
F1 Score: 0.6250
AUROC: 0.9401


Epoch 20/100
Loss: 0.1584
Accuracy: 0.9434
Precision: 0.7907
Recall: 0.5000
F1 Score: 0.6126
AUROC: 0.9357


Epoch 30/100
Loss: 0.1638
Accuracy: 0.9421
Precision: 0.7857
Recall: 0.4853
F1 Score: 0.6000
AUROC: 0.9314


Epoch 40/100
Loss: 0.1754
Accuracy: 0.9421
Precision: 0.8000
Recall: 0.4706
F1 Score: 0.5926
AUROC: 0.9243


Epoch 50/100
Loss: 0.1770
Accuracy: 0.9408
Precision: 0.7674
Recall: 0.4853
F1 Score: 0.5946
AUROC: 0.9249


Epoch 60/100
Loss: 0.1904
Accuracy: 0.9395
Precision: 0.7619
Recall: 0.4706
F1 Score: 0.5818
AUROC: 0.9193


Epoch 70/100
Loss: 0.1958
Accuracy: 0.9395
Precision: 0.7619
Recall: 0.4706
F1 Score: 0.5818
AUROC: 0.9181


Epoch 80/100
Loss: 0.2078
Accuracy: 0.9368
Precision: 0.7174
Recall: 0.4853
F1 Score: 0.5789
AUROC: 0.9155


Epoch 90/100
Loss: 0.2111
Accuracy: 0.9355
Precision: 0.7111
Recall: 0.4706
F1 Score: 0.5664
AUROC: 0.9148


Epoch 100/100
Loss